### 02 - Text - Preprocessing - Pitt

In this notebook, we will pre-process text files in Pitt set and split them into train / dev / test set with text.

It will read split id files (`pitt_train.csv`, `pitt_dev.csv`, `pitt_test.csv`) and map the text accordingly. 
Then it will export files with id, text and labels to `pitt_train_txt.csv`, `pitt_dev_txt.csv`, `pitt_test_txt.csv` for model training

In [19]:
import pandas as pd

In [20]:
# please point to correct downloaded transcribed folders of the original dataset
data_path = "../../../data/" 

fname_train = data_path + "pitt/pitt_train.csv"
fname_dev = data_path + "pitt/pitt_dev.csv"
fname_test = data_path + "pitt/pitt_test.csv"
fname_pitt_text = data_path + "pitt/text_pitt.csv"

In [21]:
train_df = pd.read_csv(fname_train)
dev_df = pd.read_csv(fname_dev)
test_df = pd.read_csv(fname_test)
text_df = pd.read_csv(fname_pitt_text)

In [22]:
# quick look at the dataframes
print("Train len =", len(train_df))
train_df['id'] = train_df['id'].astype(str)
train_df.head()

Train len = 174


,id,target,age_of_first_test,sex,less_than_65
0,714,ad,79,male,False
1,466,ad,78,female,False
2,511,ad,75,female,False
3,609,ad,70,female,False
4,504,ad,78,male,False


In [23]:
print("Dev len =", len(dev_df))
dev_df['id'] = dev_df['id'].astype(str)
dev_df.head()

Dev len = 59


,id,target,age_of_first_test,sex,less_than_65
0,234,ad,66,male,False
1,256,cn,67,male,False
2,497,ad,70,female,False
3,264,ad,65,female,False
4,297,cn,64,male,True


In [24]:
print("Test len =", len(test_df))
test_df['id'] = test_df['id'].astype(str)
test_df.head()

Test len = 59


,id,target,age_of_first_test,sex,less_than_65
0,358,ad,60,female,True
1,34,cn,70,male,False
2,42,cn,56,female,True
3,650,ad,87,female,False
4,332,cn,50,male,True


In [25]:
print("Pitt text len =", len(text_df))
# map file_id to id
text_df["id"] = text_df["file_id"].apply(lambda fid: str(fid).split("-")[0].strip()) # e.g. 001-0 -> 001
text_df.head()

Pitt text len = 549


,file_id,target,text,mmse,id
0,001-0,AD,mhm .\nalright .\nthere's &-um a young boy tha...,18.0,001
1,001-2,AD,mhm .\nthere's a young boy &-uh going in a coo...,11.0,001
2,003-0,AD,here's a cookie jar .\nand the lid is off the ...,20.0,003
3,005-0,AD,the boy is slipping off the stool .\nhe's tryi...,23.0,005
4,005-2,AD,okay ‡ he's fallin(g) off a chair .\nshe's &-...,19.0,005


In [26]:
def get_split_name(id):
    """
        return split name (train/dev/set) given id
    """
    result = "train" # default
    # remove leading zero, to be compatible with the format of split instruction csv files
    id = str(id).lstrip("0")  
    # id = str(id)
    # print("id =", id)
    if len(dev_df.query("id == @id")) > 0:
        result = "dev"
    elif len(test_df.query("id == @id")) > 0:
        result = "test"
        
    return result

assert get_split_name("714") == "train"
assert get_split_name("466") == "train"
assert get_split_name("511") == "train"

assert get_split_name("234") == "dev"
assert get_split_name("256") == "dev"
assert get_split_name("497") == "dev"


assert get_split_name("358") == "test"
assert get_split_name("034") == "test"
assert get_split_name("332") == "test"

print("Passed")

Passed


In [27]:
# debug
id = "234"
dev_df.query("id == 234")

,id,target,age_of_first_test,sex,less_than_65


In [28]:
# apply to text df
text_df["split_name"] = text_df["id"].apply(lambda id: get_split_name(id))
text_df.head()

,file_id,target,text,mmse,id,split_name
0,001-0,AD,mhm .\nalright .\nthere's &-um a young boy tha...,18.0,001,test
1,001-2,AD,mhm .\nthere's a young boy &-uh going in a coo...,11.0,001,test
2,003-0,AD,here's a cookie jar .\nand the lid is off the ...,20.0,003,train
3,005-0,AD,the boy is slipping off the stool .\nhe's tryi...,23.0,005,train
4,005-2,AD,okay ‡ he's fallin(g) off a chair .\nshe's &-...,19.0,005,train


In [29]:
print(text_df.query("id == '001'"))

  file_id target                                               text  mmse  \
0   001-0     AD  mhm .\nalright .\nthere's &-um a young boy tha...  18.0   
1   001-2     AD  mhm .\nthere's a young boy &-uh going in a coo...  11.0   

    id split_name  
0  001       test  
1  001       test  


In [30]:
assert text_df.query("id == '609'")["split_name"].iloc[0] == "train"
assert text_df.query("id == '504'")["split_name"].iloc[0] == "train"


assert text_df.query("id == '264'")["split_name"].iloc[0] == "dev"
assert text_df.query("id == '297'")["split_name"].iloc[0] == "dev"

assert text_df.query("id == '650'")["split_name"].iloc[0] == "test"
assert text_df.query("id == '332'")["split_name"].iloc[0] == "test"
print("Passed")

Passed


In [31]:
# split to 3 files
train_fname_out = data_path + "pitt/pitt_train_txt.csv"
dev_fname_out = data_path + "pitt/pitt_dev_txt.csv"
test_fname_out = data_path + "pitt/pitt_test_txt.csv"

txt_train_df = text_df.query("split_name == 'train'")
txt_dev_df = text_df.query("split_name == 'dev'")
txt_test_df = text_df.query("split_name == 'test'")

In [32]:
# more assert test to make sure no id leakage between train / dev / test set
train_ids = set(txt_train_df["id"].tolist())
dev_ids = set(txt_dev_df["id"].tolist())
test_ids = set(txt_test_df["id"].tolist())


assert len(train_ids.intersection(dev_ids)) == 0
assert len(train_ids.intersection(test_ids)) == 0
assert len(dev_ids.intersection(test_ids)) == 0

print("Passed")

Passed


In [33]:
print("Train count =", len(txt_train_df))
assert len(txt_train_df) == 337
txt_train_df.head()

Train count = 337


,file_id,target,text,mmse,id,split_name
2,003-0,AD,here's a cookie jar .\nand the lid is off the ...,20.0,003,train
3,005-0,AD,the boy is slipping off the stool .\nhe's tryi...,23.0,005,train
4,005-2,AD,okay ‡ he's fallin(g) off a chair .\nshe's &-...,19.0,005,train
5,007-1,AD,xxx .\ncan I look at it and tell you ?\noh oka...,19.0,007,train
6,007-3,AD,well the girl is telling the boy to get the co...,15.0,007,train


In [34]:
print("Dev count =", len(txt_dev_df))
assert len(txt_dev_df) == 115
txt_dev_df.head()

Dev count = 115


,file_id,target,text,mmse,id,split_name
13,016-0,AD,guy or a boy reaching for a cookie jar &-uh ti...,28.0,016,dev
14,016-1,AD,uh the water's spilling over .\ndrying the dis...,NaN,016,dev
15,016-3,AD,a boy is reaching for the cookie jar and tripp...,NaN,016,dev
16,016-4,AD,well the little girl wants the cookie that the...,NaN,016,dev
26,033-0,AD,okay .\napparently a mother and a daughter and...,26.0,033,dev


In [35]:
print("Test count =", len(txt_test_df))
assert len(txt_test_df) == 97
txt_test_df.head()

Test count = 97


,file_id,target,text,mmse,id,split_name
0,001-0,AD,mhm .\nalright .\nthere's &-um a young boy tha...,18.0,001,test
1,001-2,AD,mhm .\nthere's a young boy &-uh going in a coo...,11.0,001,test
18,023-0,AD,(..) well I see the mother .\nwhat's she doin(...,16.0,023,test
19,023-2,AD,well you see <the little> the little boy stea...,3.0,023,test
33,039-0,AD,okay we see &-um first of all &-uh the sink is...,28.0,039,test


In [36]:
# write to csv files
print("Writing to csv files...")
txt_train_df.to_csv(train_fname_out,
                    index=False
                   )
txt_dev_df.to_csv(dev_fname_out,
                  index=False
                 )
txt_test_df.to_csv(test_fname_out,
                   index=False
                   )

print("Completed")

Writing to csv files...
Completed
